# 01 · Lectura, comprensión e integración de datos

En este notebook vamos a unir los datos de clientes, cuentas, tarjetas y movimientos. La idea es obtener una tabla con una fila por cliente para usarla en los siguientes notebooks.

- `client` → `disp` por `client_id`.
- `disp` → `account` por `account_id`.
- `card` se resume por `disp_id` antes de unir.
- `trans` se resume por `account_id` antes de unir.

Comparamos las características de los clientes con y sin tarjeta para conocer mejor ambos grupos. Tener una tarjeta registrada no indica si el cliente cumple las condiciones para recibir crédito. Para calcular la edad y la antigüedad usamos como referencia el 31/12/1998.

**Salida:** `intermedios/01_base_clientes.csv` y `resultados/01_resumen_origenes.csv`.

## 1. Librerías y rutas

In [8]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')

def localizar_proyecto():
    # Funciona al abrir Jupyter desde la carpeta principal o desde notebooks/.
    for ruta in [Path.cwd(), *Path.cwd().parents]:
        if (ruta / 'datos' / 'Caso Berka.zip').is_file():
            return ruta
    raise FileNotFoundError(
        'No se encontró datos/Caso Berka.zip. Abre Jupyter desde Berka_Equipo/.'
    )

RAIZ = localizar_proyecto()
INTERMEDIOS = RAIZ / 'intermedios'
RESULTADOS = RAIZ / 'resultados'
GRAFICOS = RESULTADOS / 'graficos'
for carpeta in [INTERMEDIOS, RESULTADOS, GRAFICOS]:
    carpeta.mkdir(parents=True, exist_ok=True)
print('Proyecto:', RAIZ.resolve())

from zipfile import ZipFile

Proyecto: C:\Users\Florencia\Desktop\Modulo 3\practica\Berka_Equipo


## 2. Inventario de las ocho tablas del Banco Berka
Revisamos las ocho tablas disponibles. Usamos cinco para el análisis y también registramos la cantidad de filas y columnas de `loan`, `order` y `district`. De `trans` leemos solo las columnas necesarias.

In [9]:
valores_nulos = ['', ' ', 'NA', 'NULL', 'NaN']
with ZipFile(RAIZ / 'datos' / 'Caso Berka.zip') as archivo:
    def leer_tabla(nombre, **kwargs):
        with archivo.open(f'data/{nombre}.asc') as origen:
            return pd.read_csv(origen, sep=';', na_values=valores_nulos,
                               low_memory=False, **kwargs)

    df_client = leer_tabla('client', dtype={'birth_number': 'string'})
    df_disp = leer_tabla('disp')
    df_account = leer_tabla('account')
    df_card = leer_tabla('card')
    df_trans = leer_tabla('trans', usecols=['trans_id', 'account_id',
                                         'type', 'amount', 'balance'])
    otros = {nombre: leer_tabla(nombre) for nombre in ['loan','order','district']}

    # Cantidad real de columnas del archivo (en trans se leen solo cinco).
    columnas_origen = {}
    for nombre in ['client','disp','account','card','trans','loan','order','district']:
        with archivo.open(f'data/{nombre}.asc') as f:
            columnas_origen[nombre] = len(f.readline().decode('utf-8').strip().split(';'))

fuentes = {'client': df_client, 'disp': df_disp, 'account': df_account,
           'card': df_card, 'trans': df_trans, **otros}
resumen_origenes = pd.DataFrame([
    {'tabla': nombre, 'filas': len(df), 'columnas_origen': columnas_origen[nombre],
     'columnas_leidas': df.shape[1], 'duplicados_exactos': int(df.duplicated().sum()),
     'nulos_celdas_leidas': int(df.isna().sum().sum())}
    for nombre, df in fuentes.items()
])
print(resumen_origenes.to_string(index=False))
resumen_origenes.to_csv(RESULTADOS / '01_resumen_origenes.csv', index=False,
                       encoding='utf-8-sig')

   tabla   filas  columnas_origen  columnas_leidas  duplicados_exactos  nulos_celdas_leidas
  client    5369                3                3                   0                    0
    disp    5369                4                4                   0                    0
 account    4500                4                4                   0                    0
    card     892                4                4                   0                    0
   trans 1056320               10                5                   0                    0
    loan     682                7                7                   0                    0
   order    6471                6                6                   0                 1379
district      77               16               16                   0                    0


## 3. Inspección inicial (`head`, `dtypes`, `describe`)

In [10]:
print('CLIENTES')
display(df_client.head())
print(df_client.dtypes)
print('\nDISPOSICIONES')
display(df_disp.head())
print('\nCUENTAS')
display(df_account.head())
print('\nTARJETAS')
display(df_card.head())
print('\nTRANSACCIONES (campos seleccionados)')
display(df_trans.head())

CLIENTES


,client_id,birth_number,district_id
0,1,706213,18
1,2,450204,1
2,3,406009,1
3,4,561201,5
4,5,605703,5


client_id                int64
birth_number    string[python]
district_id              int64
dtype: object

DISPOSICIONES


,disp_id,client_id,account_id,type
0,1,1,1,OWNER
1,2,2,2,OWNER
2,3,3,2,DISPONENT
3,4,4,3,OWNER
4,5,5,3,DISPONENT



CUENTAS


,account_id,district_id,frequency,date
0,576,55,POPLATEK MESICNE,930101
1,3818,74,POPLATEK MESICNE,930101
2,704,55,POPLATEK MESICNE,930101
3,2378,16,POPLATEK MESICNE,930101
4,2632,24,POPLATEK MESICNE,930102



TARJETAS


,card_id,disp_id,type,issued
0,1005,9285,classic,931107 00:00:00
1,104,588,classic,940119 00:00:00
2,747,4915,classic,940205 00:00:00
3,70,439,classic,940208 00:00:00
4,577,3687,classic,940215 00:00:00



TRANSACCIONES (campos seleccionados)


,trans_id,account_id,type,amount,balance
0,695247,2378,PRIJEM,700.0,700.0
1,171812,576,PRIJEM,900.0,900.0
2,207264,704,PRIJEM,1000.0,1000.0
3,1117247,3818,PRIJEM,600.0,600.0
4,579373,1972,PRIJEM,400.0,400.0


## 4. Transformación de columnas y fechas
En `birth_number`, el mes de nacimiento de las mujeres tiene 50 sumado. Corregimos ese valor para obtener la fecha de nacimiento y calculamos la edad al 31/12/1998.

In [ ]:
FECHA_REFERENCIA = pd.Timestamp('1998-12-31')

nac = df_client['birth_number'].astype('string').str.zfill(6)
anio = 1900 + nac.str[:2].astype(int)
mes_codificado = nac.str[2:4].astype(int)
mes = np.where(mes_codificado > 50, mes_codificado - 50, mes_codificado)
dia = nac.str[4:6].astype(int)

df_client['sexo'] = np.where(mes_codificado > 50, 'F', 'M')
df_client['fecha_nacimiento'] = pd.to_datetime(
    {'year': anio, 'month': mes, 'day': dia}, errors='coerce')
df_client['edad_1998'] = (
    FECHA_REFERENCIA.year - df_client['fecha_nacimiento'].dt.year
    - ((FECHA_REFERENCIA.month < df_client['fecha_nacimiento'].dt.month)
       | ((FECHA_REFERENCIA.month == df_client['fecha_nacimiento'].dt.month)
          & (FECHA_REFERENCIA.day < df_client['fecha_nacimiento'].dt.day)))
)

df_account['fecha_apertura'] = pd.to_datetime(
    df_account['date'].astype(str).str.zfill(6), format='%y%m%d', errors='coerce')
df_account['antiguedad_cuenta_anios'] = (
    FECHA_REFERENCIA - df_account['fecha_apertura']).dt.days / 365.25

df_card['fecha_emision'] = pd.to_datetime(
    df_card['issued'], format='%y%m%d %H:%M:%S', errors='coerce')

print('Nacimientos no válidos:', int(df_client['fecha_nacimiento'].isna().sum()))
print('Aperturas no válidas:', int(df_account['fecha_apertura'].isna().sum()))
print('Emisiones no válidas:', int(df_card['fecha_emision'].isna().sum()))
display(df_client[['client_id','sexo','fecha_nacimiento','edad_1998']].head())

Nacimientos no válidos: 0
Aperturas no válidas: 0
Emisiones no válidas: 0


,client_id,sexo,fecha_nacimiento,edad_1998
0,1,F,1970-12-13,28
1,2,M,1945-02-04,53
2,3,F,1940-10-09,58
3,4,M,1956-12-01,42
4,5,F,1960-07-03,38


## 5. Agregaciones con `groupby` antes de los `merge`
Una cuenta puede tener muchas transacciones. Para evitar que un cliente aparezca en varias filas, primero resumimos las transacciones por `account_id` y las tarjetas por `disp_id`.

El `saldo_promedio` se calcula con los saldos registrados después de cada transacción. Por eso, no es un promedio diario.

In [12]:
df_trans['ingreso'] = np.where(df_trans['type'].eq('PRIJEM'),
                               df_trans['amount'], 0.0)
df_trans['egreso'] = np.where(df_trans['type'].isin(['VYDAJ','VYBER']),
                              df_trans['amount'], 0.0)
print('Tipos de transacción:')
print(df_trans.groupby('type').size())

df_trans_agg = df_trans.groupby('account_id', as_index=False).agg(
    num_transacciones=('trans_id', 'count'),
    importe_ingresos=('ingreso', 'sum'),
    importe_egresos=('egreso', 'sum'),
    importe_medio_transaccion=('amount', 'mean'),
    saldo_promedio=('balance', 'mean'))

df_card_agg = df_card.groupby('disp_id', as_index=False).agg(
    n_tarjetas=('card_id', 'nunique'),
    fecha_primera_tarjeta=('fecha_emision', 'min'))

assert df_trans_agg['account_id'].is_unique
assert df_card_agg['disp_id'].is_unique
print('Cuentas con movimientos:', len(df_trans_agg))
print('Disposiciones con tarjeta:', len(df_card_agg))

Tipos de transacción:
type
PRIJEM    405083
VYBER      16666
VYDAJ     634571
dtype: int64
Cuentas con movimientos: 4500
Disposiciones con tarjeta: 892


## 6. Integración de cinco fuentes con validaciones

In [13]:
# Se conserva cada cliente aunque no tenga tarjeta: LEFT JOIN.
df_base = (
    df_client.rename(columns={'district_id':'district_id_cliente'})
    .merge(df_disp.rename(columns={'type':'tipo_disposicion'}),
           on='client_id', how='left', validate='one_to_one')
    .merge(df_account.rename(columns={
           'district_id':'district_id_cuenta',
           'date':'fecha_apertura_codificada',
           'frequency':'frecuencia'}),
           on='account_id', how='left', validate='many_to_one')
    .merge(df_card_agg, on='disp_id', how='left', validate='one_to_one')
    .merge(df_trans_agg, on='account_id', how='left', validate='many_to_one')
)

df_base['n_tarjetas'] = df_base['n_tarjetas'].fillna(0).astype(int)
df_base['tiene_tarjeta'] = (df_base['n_tarjetas'] > 0).astype(int)

assert len(df_base) == len(df_client), 'Se multiplicaron o perdieron clientes.'
assert df_base['client_id'].is_unique, 'La granularidad dejó de ser cliente.'
assert df_base['account_id'].notna().all(), 'Hay cuentas sin enlazar.'
assert df_base['tiene_tarjeta'].isin([0,1]).all()
print('Base integrada:', df_base.shape)
print('Clientes únicos:', df_base['client_id'].nunique())
print('Distribución de Y:')
print(df_base['tiene_tarjeta'].value_counts())
display(df_base[['client_id','account_id','edad_1998',
                 'frecuencia','saldo_promedio','tiene_tarjeta']].head())

Base integrada: (5369, 22)
Clientes únicos: 5369
Distribución de Y:
tiene_tarjeta
0    4477
1     892
Name: count, dtype: int64


,client_id,account_id,edad_1998,frecuencia,saldo_promedio,tiene_tarjeta
0,1,1,28,POPLATEK MESICNE,15994.008368,0
1,2,2,53,POPLATEK MESICNE,36540.704393,0
2,3,2,58,POPLATEK MESICNE,36540.704393,0
3,4,3,42,POPLATEK MESICNE,26973.506838,0
4,5,3,38,POPLATEK MESICNE,26973.506838,0


## 7. Guardar resultado y responder preguntas de 3.2
Identificamos ocho tablas y usamos cinco para construir la base. Después de las uniones, cada cliente ocupa una fila. La variable Y indica si tiene una tarjeta registrada.

Algunas cuentas son compartidas, así que sus movimientos y saldos pueden aparecer en más de un cliente. Además, los datos no indican si un cliente es apto para recibir crédito.

In [7]:
# Los formatos de fecha se conservan como fechas ISO al exportar.
ruta_base = INTERMEDIOS / '01_base_clientes.csv'
df_base.to_csv(ruta_base, index=False, encoding='utf-8-sig',
               date_format='%Y-%m-%d')
print('Archivo de integración:', ruta_base)
print('Nulos por columna:')
print(df_base.isna().sum().sort_values(ascending=False).to_string())

Archivo de integración: c:\Users\Florencia\Desktop\Modulo 3\practica\Berka_Equipo\intermedios\01_base_clientes.csv
Nulos por columna:
fecha_primera_tarjeta        4477
client_id                       0
district_id_cliente             0
sexo                            0
fecha_nacimiento                0
birth_number                    0
edad_1998                       0
disp_id                         0
tipo_disposicion                0
account_id                      0
frecuencia                      0
fecha_apertura_codificada       0
fecha_apertura                  0
district_id_cuenta              0
antiguedad_cuenta_anios         0
n_tarjetas                      0
num_transacciones               0
importe_ingresos                0
importe_egresos                 0
importe_medio_transaccion       0
saldo_promedio                  0
tiene_tarjeta                   0
